# Paper results

This report reads the per-task results of the two finalists reported in the PASSAGE paper, released under `src/passage-eval/results/paper-results/`: ConvNeXt-V2 Atto segmentation with BCE and RepViT-M1 regression with the Bellman auxiliary and the V_bridge decoder, at 256×256 with the `elevation` objective, for three training seeds.

The report downloads no data and runs no training, inference or measurement.

In [ ]:
# Papermill overrides this cell before any imports or derived paths.
resolution = 256
project_path = None  # Optional checkout root for execution in an external staging directory.
dataset_path = None  # Not read: this report uses released result files only.
dataset_repo = "thalesgroup/passage"
dataset_revision = None
output_dir = "outputs/notebook-reports"
result_root = "src/passage-eval/results/paper-results"  # Released per-task results of the paper finalists.

In [ ]:
from pathlib import Path
import sys
import json

# Discover repository root from the execution directory without changing it.
project_root = Path(project_path).resolve() if project_path else next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
if (project_root / "src/passage").is_dir():
    sys.path.insert(0, str(project_root / "src"))
# Otherwise use the installed PASSAGE package; no private checkout is required.
from passage.notebook_reports import resolve_path, show_records, write_report
from IPython.display import display, Markdown
results_path = resolve_path(result_root, project_root)
output_path = resolve_path(output_dir, project_root)

In [ ]:
METRICS = ("cost_regret_median", "cost_regret_p90", "path_assd", "path_hausdorff_p95",
           "success_rate", "obstacle_violation_rate")
runs = {}
for path in sorted(results_path.rglob("metrics.json")):
    payload = json.loads(path.read_text())
    if payload.get("resolution") != resolution:
        continue
    key = (payload["method_name"], payload["seed"])
    if key in runs and runs[key]["payload"] != payload:
        raise ValueError(f"Inconsistent repeated records for {key}: {path}")
    runs.setdefault(key, {"payload": payload, "directories": 0})["directories"] += 1
records = [{"method_name": method, "seed": seed, "cost_model": run["payload"]["cost_model"],
            "n_samples": run["payload"]["metrics"]["n_samples"], "directories": run["directories"],
            **{name: run["payload"]["metrics"].get(name) for name in METRICS}}
           for (method, seed), run in sorted(runs.items())]
show_records(records)

In [ ]:
summary = []
for method in sorted({record["method_name"] for record in records}):
    rows = [record for record in records if record["method_name"] == method]
    entry = {"method_name": method, "seeds": len(rows)}
    for name in METRICS:
        values = [row[name] for row in rows if row[name] == row[name]]
        entry[name] = sum(values) / len(values) if values else None
    summary.append(entry)
show_records(summary)
display(Markdown("Seed means of the reported per-seed metrics."))

In [ ]:
report = {"schema_version": 1, "kind": "paper_results", "resolution": resolution,
          "leaderboard_entries": False,
          "records": records, "summary": summary}
report_path = write_report(report, output_path, f"02_paper_results-{resolution}.json")
shown = report_path.relative_to(project_root) if report_path.is_relative_to(project_root) else report_path.name
display(Markdown(f"Report receipt: `{shown}`."))